In [14]:
import pandas as pd

# Load files
discovery = pd.read_csv('GSE192902_counts_Discovery_postQC.csv')
validation1 = pd.read_csv('GSE192902_counts_Validation1_postQC.csv')
validation2 = pd.read_csv('GSE192902_counts_Validation2_postQC.csv')

def deduplicate_by_gene_name(df, file_name):
    # Find duplicates by gene_name
    duplicates = df[df['gene_name'].duplicated(keep=False)]
    num_duplicates = df['gene_name'].duplicated().sum()
    print(f"\nNumber of duplicate gene names in {file_name}: {num_duplicates}")
    print(f"Duplicate gene names in {file_name}:")
    print(duplicates['gene_name'].unique())

    # Deduplicate by summing counts for duplicated gene names
    deduplicated_df = df.groupby('gene_name', as_index=False).sum(numeric_only=True)
    return deduplicated_df

# Deduplicate each file by gene_name only
discovery_dedup = deduplicate_by_gene_name(discovery, 'Discovery')
validation1_dedup = deduplicate_by_gene_name(validation1, 'Validation1')
validation2_dedup = deduplicate_by_gene_name(validation2, 'Validation2')

# Merge the files on gene_name only (no gene_num anymore)
merged = discovery_dedup[['gene_name']].merge(
    validation1_dedup,
    on='gene_name',
    how='left'
).merge(
    validation2_dedup,
    on='gene_name',
    how='left'
)

# Fill missing values with 0
merged.fillna(0, inplace=True)

# Perform CPM normalisation
counts_only = merged.drop(columns=['gene_name'])
counts_sum = counts_only.sum(axis=0)
cpm = counts_only.divide(counts_sum, axis=1) * 1e6

merged_cpm = pd.concat([merged[['gene_name']], cpm], axis=1)
# Load the SraRunTable
sra_run_table = pd.read_csv('GSE192902_SraRunTable.csv')

# Extract the relevant column names from the Library Name column
library_names = sra_run_table['Library Name'].dropna().unique()

# Keep gene_name columns, plus only those Library Names present in merged_cpm
cols_to_keep = ['gene_name'] + [col for col in merged_cpm.columns if col in library_names]

# Filter merged_cpm columns to these
filtered_merged_cpm = merged_cpm[cols_to_keep]

# Join filtered counts with disease info from sra_run_table on Library Name (column names)
# Create a mapping Library Name -> disease
libname_to_disease = sra_run_table.set_index('Library Name')['disease'].to_dict()

# Extract sample columns (excluding gene_name and gene_num)
sample_cols = [col for col in filtered_merged_cpm.columns if col not in ['gene_name']]

# Create masks for samples based on disease
pre_eclampsia_samples = [col for col in sample_cols if libname_to_disease.get(col, '').lower() == 'pre-eclampsia']
severe_pre_eclampsia_samples = [col for col in sample_cols if libname_to_disease.get(col, '').lower() == 'severe pre-eclampsia']

# Identify normotensive (unlisted) samples
all_count_samples = set([col for col in merged_cpm.columns if col != 'gene_name'])
listed_samples = set(library_names)  # from SRA run table

# Samples that appear in counts but are not listed in the metadata
normotensive_samples = sorted(list(all_count_samples - listed_samples))

print(f"Total normotensive samples detected: {len(normotensive_samples)}")
print(f"Example normotensive samples: {normotensive_samples[:10]}")



Number of duplicate gene names in Discovery: 51
Duplicate gene names in Discovery:
['none' 'Y_RNA' 'U1' '5_8S_rRNA']

Number of duplicate gene names in Validation1: 51
Duplicate gene names in Validation1:
['none' 'Y_RNA' 'U1' '5_8S_rRNA']

Number of duplicate gene names in Validation2: 51
Duplicate gene names in Validation2:
['none' 'Y_RNA' 'U1' '5_8S_rRNA']
Total normotensive samples detected: 145
Example normotensive samples: ['103651_1', '108445_1', '121329_2', '122540_2', '133532_1', '138919_1', '141218_1', '141956_2', '149589_1', '161584_1']


In [12]:
import pandas as pd
import re

# Load the GTF file and build the gene_id to gene_name mapping
gtf_file = "../gencode.v46.annotation.gtf"
gene_mapping = {}

with open(gtf_file, 'r') as file:
    for line in file:
        if line.startswith('#'):
            continue
        fields = line.strip().split('\t')
        if fields[2] != 'gene':
            continue
        attributes = fields[8]
        gene_id_match = re.search('gene_id "([^"]+)"', attributes)
        gene_name_match = re.search('gene_name "([^"]+)"', attributes)
        if gene_id_match and gene_name_match:
            gene_id = gene_id_match.group(1).split('.')[0]  # Remove version number
            gene_name = gene_name_match.group(1)
            gene_mapping[gene_id] = gene_name

print(f"Loaded {len(gene_mapping)} gene mappings from GTF.")

# Function to process each file
def process_file(file_path, label, output_file):
    print(f"\nProcessing {label}...")

    # Load counts file
    df = pd.read_csv(file_path)

    # Replace gene names using gene_num
    df['gene_name'] = df['gene_num'].map(gene_mapping).fillna(df['gene_name'])

    # Drop 'none' genes
    df_filtered = df[df['gene_name'] != 'none'].copy()
    print(f"Remaining genes after dropping 'none': {df_filtered.shape[0]}")

    # CPM normalization
    counts_only = df_filtered.drop(['gene_name', 'gene_num'], axis=1)
    library_sizes = counts_only.sum(axis=0)
    cpm = counts_only.div(library_sizes, axis=1) * 1e6
    cpm.insert(0, 'gene_name', df_filtered['gene_name'].values)

    # --- NEW STEP: keep sample_3, or fallback to sample_2, then sample_1 for each patient ---
    sample_to_patient = {s: s.split('_')[0] for s in counts_only.columns}

    selected_samples = []
    sample_choice_log = []  # to record which sample was chosen per patient

    for patient in sorted(set(sample_to_patient.values())):
        # Find all samples belonging to this patient
        patient_samples = [s for s, p in sample_to_patient.items() if p == patient]

        # Try selecting in priority order: _3 > _2 > _1
        sample3 = [s for s in patient_samples if s.endswith('_3')]
        sample2 = [s for s in patient_samples if s.endswith('_2')]
        sample1 = [s for s in patient_samples if s.endswith('_1')]

        if sample3:
            chosen = sample3[0]
            reason = "_3"
        elif sample2:
            chosen = sample2[0]
            reason = "_2"
        elif sample1:
            chosen = sample1[0]
            reason = "_1"
        else:
            # Skip patient if none of the preferred samples exist
            continue

        selected_samples.append(chosen)
        sample_choice_log.append((patient, chosen, reason))

    removed_samples = len(counts_only.columns) - len(selected_samples)

    # Subset CPM to only the chosen samples
    cpm = cpm[['gene_name'] + selected_samples]

    # Deduplicate by summing CPMs for genes with the same name
    deduplicated_cpm = cpm.groupby('gene_name').sum().reset_index()

    print(f"Final deduplicated matrix shape: {deduplicated_cpm.shape}")
    print(f"Unique patients saved: {len(selected_samples)}")
    print(f"Samples removed: {removed_samples}")

    # Optional: summary of what was chosen
    from collections import Counter
    summary = Counter([r for _, _, r in sample_choice_log])
    print(f"Selected sample types: {dict(summary)}")  # e.g. {'_3': 50, '_2': 12, '_1': 3}

    # Save deduplicated file
    deduplicated_cpm.to_csv(output_file, sep='\t', index=False)
    print(f"Saved deduplicated CPM-normalized file: {output_file}")


# Process Stanford file
process_file(
    'GSE192902_counts_Stanford_preQC.csv',
    label='Stanford',
    output_file='Stanford_CPM_UniquePatient_SpecificSample_deduplicated.txt'
)

# Process Validation2 file
process_file(
    'GSE192902_counts_Validation2_preQC.csv',
    label='Validation2',
    output_file='Validation2_CPM_UniquePatient_SpecificSample_deduplicated.txt'
)

Loaded 63086 gene mappings from GTF.

Processing Stanford...
Remaining genes after dropping 'none': 57565
Final deduplicated matrix shape: (56361, 110)
Unique patients saved: 109
Samples removed: 206
Selected sample types: {'_3': 82, '_2': 21, '_1': 6}
Saved deduplicated CPM-normalized file: Stanford_CPM_UniquePatient_SpecificSample_deduplicated.txt

Processing Validation2...
Remaining genes after dropping 'none': 57565
Final deduplicated matrix shape: (56361, 88)
Unique patients saved: 87
Samples removed: 2
Selected sample types: {'_1': 54, '_2': 33}
Saved deduplicated CPM-normalized file: Validation2_CPM_UniquePatient_SpecificSample_deduplicated.txt


In [15]:
import pandas as pd

# Load CPM deduplicated files
stanford_cpm = pd.read_csv('Stanford_CPM_UniquePatient_SpecificSample_deduplicated.txt', sep='\t')
validation2_cpm = pd.read_csv('Validation2_CPM_UniquePatient_SpecificSample_deduplicated.txt', sep='\t')

# Get available columns (samples) in each file
stanford_cols = set(stanford_cpm.columns.tolist())
validation2_cols = set(validation2_cpm.columns.tolist())

# Function to extract samples and report missing ones
def extract_samples(sample_list, label, output_file):
    sample_set = set(sample_list)
    
    stanford_samples = [s for s in sample_set if s in stanford_cols]
    validation2_samples = [s for s in sample_set if s in validation2_cols]
    
    found_samples = set(stanford_samples + validation2_samples)
    missing_samples = sample_set - found_samples

    dfs = []
    if stanford_samples:
        sub_df = stanford_cpm[['gene_name'] + stanford_samples]
        dfs.append(sub_df)
    if validation2_samples:
        sub_df = validation2_cpm[['gene_name'] + validation2_samples]
        dfs.append(sub_df)
    
    # Merge on gene_name
    if len(dfs) == 2:
        merged_df = pd.merge(dfs[0], dfs[1], on='gene_name', how='outer')
    else:
        merged_df = dfs[0]
    
    # Save to file
    merged_df.to_csv(output_file, sep='\t', index=False)
    print(f"\nSaved {label} counts to {output_file}")
    print(f"Number of samples in {label} list: {len(sample_list)}")
    print(f"Number of samples found: {len(found_samples)}")
    print(f"Number of samples missing: {len(missing_samples)}")

    if missing_samples:
        print(f"Missing samples from {label}: {sorted(missing_samples)}")

# Extract pre-eclampsia samples
extract_samples(pre_eclampsia_samples, 'Pre-eclampsia', 'GSE192902_Pre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample.txt')

# Extract severe pre-eclampsia samples
extract_samples(severe_pre_eclampsia_samples, 'Severe Pre-eclampsia', 'GSE192902_SeverePre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample.txt')

# Extract normotensive pregnancies
extract_samples(normotensive_samples, 'Normotensive pregnancies', 'GSE192902_Normotensive_preQC_filtered_counts_UniquePatient_SpecificSample.txt')


Saved Pre-eclampsia counts to GSE192902_Pre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample.txt
Number of samples in Pre-eclampsia list: 30
Number of samples found: 17
Number of samples missing: 13
Missing samples from Pre-eclampsia: ['267586_1', '349246_1', '546738_2', '546738_2.1', '546738_4', '631489_1', '631489_2', '631489_4', '674547_1', '674547_2', '813805_1', '813805_2', '813805_4']

Saved Severe Pre-eclampsia counts to GSE192902_SeverePre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample.txt
Number of samples in Severe Pre-eclampsia list: 20
Number of samples found: 16
Number of samples missing: 4
Missing samples from Severe Pre-eclampsia: ['569467_2.1', '597312_2', '597312_4', '739041_4']

Saved Normotensive pregnancies counts to GSE192902_Normotensive_preQC_filtered_counts_UniquePatient_SpecificSample.txt
Number of samples in Normotensive pregnancies list: 145
Number of samples found: 91
Number of samples missing: 54
Missing samples from Normotensive 